# Analyse du fichier Gestion Stock V4

Ce notebook analyse localement `Gestion Stock V4 - Stock Complet.csv`. Il ne se connecte pas a Odoo et ne modifie aucune donnee.

Le fichier contient quatre lignes de synthese avant l'en-tete. L'en-tete contient des libelles multilignes, des colonnes vides et deux colonnes `EUROCODE`; le chargement utilise donc les positions documentees des colonnes.

In [1]:
from pathlib import Path
import re
import unicodedata

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 160)

In [2]:
candidate_paths = [
    Path.cwd() / 'Gestion Stock V4 - Stock Complet.csv',
    Path.cwd() / 'Jobs' / 'Gestion Stock' / 'Gestion Stock V4 - Stock Complet.csv',
]
CSV_PATH = next((path for path in candidate_paths if path.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError('CSV introuvable. Lancez le notebook depuis le dossier Gestion Stock ou le depot.')
CSV_PATH = CSV_PATH.resolve()
print(f'Source: {CSV_PATH}')

Source: D:\git\rpbm\Jobs\Gestion Stock\Gestion Stock V4 - Stock Complet.csv


In [3]:
# Les quatre premieres lignes sont la synthese; la cinquieme est l'en-tete.
raw = pd.read_csv(
    CSV_PATH,
    skiprows=4,
    header=None,
    dtype=str,
    keep_default_na=False,
    engine='python',
    quotechar='"',
)
header = raw.iloc[0].tolist()
data = raw.iloc[1:].reset_index(drop=True)

# Positions zero-based dans l'export source.
column_indexes = {
    'row_id': 3, 'alternate_code': 4, 'eurocode': 5, 'source_type': 6,
    'designation': 7, 'supplier': 8, 'quantity_current': 9,
    'public_price': 10, 'purchase_price': 11, 'line_value_before_freight': 12,
    'resale_price': 13, 'total_value': 14, 'outgoing_count': 16,
    'freight': 17, 'invoice_date': 18, 'place': 19, 'exit_date': 20,
    'inventory_status': 22, 'inventory_notes': 23,
    'duplicate_eurocode': 24, 'inventory_state': 25,
}

def source_column(index):
    return data.iloc[:, index].fillna('').astype(str).str.strip()

articles = pd.DataFrame({name: source_column(index) for name, index in column_indexes.items()})
print(f'Lignes brutes apres l en-tete: {len(articles):,}')
print(f'Colonnes detectees dans l en-tete: {len(header)}')
display(articles.head(3))

Lignes brutes apres l en-tete: 10,066
Colonnes detectees dans l en-tete: 60


,row_id,alternate_code,eurocode,source_type,designation,supplier,quantity_current,public_price,purchase_price,line_value_before_freight,resale_price,total_value,outgoing_count,freight,invoice_date,place,exit_date,inventory_status,inventory_notes,duplicate_eurocode,inventory_state
0,L0001,,8308AGN,PARE BRISE,PB VT TOYOTA HI-LUX 97-05,VSF Centre,0,"83,77","63,67","0,00","140,07","0,00",1,BATEAU,11/10/2016,R122,11/07/2017,,,,
1,L0003,,6570AGNMVZ,PARE BRISE,PB VT PEUGEOT 308 II+ CAPT 13-,VSF Centre,0,"119,00","90,44","0,00","198,97","0,00",1,BATEAU,11/10/2016,R229,03/01/2018,,,,
2,L0004,,6570AGNMVZ,PARE BRISE,PB VT PEUGEOT 308 II+ CAPT 13-,VSF Centre,0,,"90,44","0,00","198,97","0,00",1,BATEAU,11/10/2016,R229,20/02/2017,,,,


In [4]:
def parse_european_number(value):
    value = str(value).strip().replace(' ', '')
    if not value:
        return None
    if ',' in value:
        value = value.replace('.', '').replace(',', '.')
    try:
        return float(value)
    except ValueError:
        return None

for column in ['quantity_current', 'public_price', 'purchase_price', 'line_value_before_freight', 'resale_price', 'total_value']:
    articles[f'{column}_num'] = articles[column].map(parse_european_number)

valid_code = articles['eurocode'].ne('') & articles['eurocode'].ne('0')
invalid_code = articles['eurocode'].isin(['-', '---', '?'])
importable_code = valid_code & ~invalid_code

summary = pd.Series({
    'lignes brutes': len(articles),
    'lignes avec EUROCODE': int(valid_code.sum()),
    'EUROCODE uniques': int(articles.loc[valid_code, 'eurocode'].nunique()),
    'references invalides': int(invalid_code.sum()),
    'references candidates import': int(articles.loc[importable_code, 'eurocode'].nunique()),
    'lignes avec quantite positive': int((articles['quantity_current_num'] > 0).sum()),
    'quantite positive totale': articles.loc[articles['quantity_current_num'] > 0, 'quantity_current_num'].sum(),
    'emplacements renseignes': int(articles['place'].ne('').sum()),
    'emplacements distincts': int(articles.loc[articles['place'].ne(''), 'place'].nunique()),
})
display(summary.to_frame('valeur'))

,valeur
lignes brutes,10066.000
lignes avec EUROCODE,10063.000
EUROCODE uniques,3292.000
references invalides,3.000
references candidates import,3289.000
lignes avec quantite positive,768.000
quantite positive totale,767.001
emplacements renseignes,9978.000
emplacements distincts,568.000


In [5]:
def normalize_key(value):
    value = unicodedata.normalize('NFD', str(value).strip().upper())
    value = ''.join(char for char in value if unicodedata.category(char) != 'Mn')
    return re.sub(r'[^A-Z0-9]+', ' ', value).strip()

def manager_category(source_type):
    key = normalize_key(source_type)
    if key == 'PARE BRISE':
        return 'Pare-brise'
    if key.startswith('LUNETTE'):
        return 'Lunette'
    if key == 'GLACES LATERALES':
        return 'Glace laterale'
    if key in {'JOINT', 'JOINTS', 'ENJOLIVEURS'}:
        return 'Joint'
    return 'Autres'

articles['target_category'] = articles['source_type'].map(manager_category)
type_summary = (
    articles.loc[valid_code]
    .groupby(['source_type', 'target_category'], dropna=False)
    .size().reset_index(name='line_count')
    .sort_values('line_count', ascending=False)
)
display(type_summary)

,source_type,target_category,line_count
19,PARE BRISE,Pare-brise,5268
11,JOINTS,Joint,2155
10,GLACES LATERALES,Glace laterale,1534
14,LUNETTES,Lunette,981
20,PARE-BRISE,Pare-brise,26
13,LEVE-VITRE,Autres,13
26,TOIT,Autres,11
17,OPTIQUE,Autres,10
2,AUTRES,Autres,10
9,FEU,Autres,8


In [6]:
# Une reference peut correspondre a plusieurs lignes, lieux ou mouvements.
duplicate_codes = (
    articles.loc[importable_code]
    .groupby('eurocode')
    .agg(line_count=('eurocode', 'size'), designations=('designation', 'nunique'), places=('place', 'nunique'))
    .query('line_count > 1')
    .sort_values('line_count', ascending=False)
)
display(duplicate_codes.head(20))

location_summary = (
    articles.loc[articles['place'].ne('')]
    .groupby('place')
    .agg(line_count=('place', 'size'), positive_quantity=('quantity_current_num', lambda values: values[values > 0].sum()))
    .sort_values('line_count', ascending=False)
)
display(location_summary.head(30))

,line_count,designations,places
eurocode,,,
7262AGSV1M,93,4,57
2741AGSVZ1P,92,10,55
7248AKMH,89,7,47
6564AGNVZ,84,5,61
7248AGS1B,81,6,47
7262ASMHT,75,3,45
6571AGRCIMVZ,71,7,53
6548AGSVZ,55,4,43
7274AGSV1C,49,4,39


,line_count,positive_quantity
place,,
Centre,1320,0.0
GALLERIA,470,14.0
GENIPA,204,7.0
CASSE,183,0.0
D2,92,0.0
Galléria,79,0.0
Tringle,61,0.0
Génipa,58,0.0
R906,53,5.0


In [7]:
quality_checks = pd.Series({
    'designations absentes': int(articles.loc[importable_code, 'designation'].eq('').sum()),
    'types absents': int(articles.loc[valid_code, 'source_type'].eq('').sum()),
    'places absents': int(articles.loc[valid_code, 'place'].eq('').sum()),
    'prix achat non numeriques': int((articles.loc[importable_code, 'purchase_price'].ne('') & articles.loc[importable_code, 'purchase_price_num'].isna()).sum()),
    'prix public non numeriques': int((articles.loc[importable_code, 'public_price'].ne('') & articles.loc[importable_code, 'public_price_num'].isna()).sum()),
    'quantites negatives': int((articles['quantity_current_num'] < 0).sum()),
    'lignes sans date de sortie': int(articles.loc[valid_code, 'exit_date'].eq('').sum()),
    'lignes inventaire pas la': int(articles['inventory_status'].str.upper().eq('PAS LA').sum()),
    'lignes inventaire erreur': int(articles['inventory_state'].str.upper().eq('ERREUR').sum()),
})
display(quality_checks.to_frame('nombre'))

stock_candidate = articles.loc[importable_code & (articles['quantity_current_num'] > 0) & articles['exit_date'].eq('')].copy()
display(stock_candidate[['eurocode', 'designation', 'quantity_current', 'place', 'inventory_status', 'inventory_state']].head(30))

,nombre
designations absentes,5
types absents,5
places absents,88
prix achat non numeriques,17
prix public non numeriques,19
quantites negatives,0
lignes sans date de sortie,769
lignes inventaire pas la,62
lignes inventaire erreur,0


,eurocode,designation,quantity_current,place,inventory_status,inventory_state
157,2726ACCMV1B,PB REFL.CITROEN C3 +CAPT 03-09,1,R203,ok,
216,7276ASMHB,JC INF.PB DACIA SANDERO/DUSTER 10-,1,J8C,ok,
304,7276ASMHB,JC INF.PB DACIA SANDERO/DUSTER 10-,1,J8C,,
417,4438RGNR5FD1M,DESC.AVD VT KIA SPORTAGE 10-,1,R226,,
423,8410ASMR,JC PB TOYOTA RAV 4 13-,1,J3F,,
426,5153ASMH,JC PB MAZDA 626 GF 97-03,1,J6F,,
442,8400LGNH3FV,DEFL.AVG VT TOYOTA YARIS III 3/5P 11-,1,T011,,
445,3562BGSHI1J,LUN.VT FORD FIESTA VI 3P 02-08,1,R336,,
512,5142AGN,PB VT MAZDA 626 GE 5PTES 92-97,1,R103,,
521,7257AKMV,JC+C PB KIT COMP. SCENIC II 03-09,1,J10C,,


## Exports optionnels

La cellule suivante est desactivee par defaut. Les exports produits sont des analyses locales, pas des fichiers d'import Odoo.

In [8]:
# EXPORT = False
# if EXPORT:
#     output_dir = CSV_PATH.parent / 'notebook_outputs'
#     output_dir.mkdir(exist_ok=True)
#     type_summary.to_csv(output_dir / 'type_summary.csv', index=False, encoding='utf-8-sig')
#     duplicate_codes.reset_index().to_csv(output_dir / 'duplicate_codes.csv', index=False, encoding='utf-8-sig')
#     location_summary.reset_index().to_csv(output_dir / 'location_summary.csv', index=False, encoding='utf-8-sig')
#     stock_candidate.to_csv(output_dir / 'stock_candidate.csv', index=False, encoding='utf-8-sig')
#     print(f'Exports ecrits dans: {output_dir}')